In [1]:
!nvidia-smi

Mon Jan 26 08:37:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -U git+https://github.com/huggingface/transformers.git

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-5q619fhl
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-5q619fhl
  Resolved https://github.com/huggingface/transformers.git to commit 1e8d1297b5a8d7702e6d8ad8e5fc38cd67cf98d9
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.6/536.6 kB 27.3 MB/s eta 0:00:00
  Created wheel for transformers: filename=transformers-5.0.0.dev0-py3-none-any.whl size=11191530 sha256=a9393ea441a264f44f10d48816bb8fd21f7e5e04c55abeb907a2cfc581ee53fd
  Stored in directory: /tmp/pip-ephem-wheel-cache-nq844b1l/wheels/54/cb/3f/83103de5575c534436d6a4686686dead458238dfaf1147e98d
Successfully built transformers
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-h

In [3]:
!pip install qwen-vl-utils accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 30.5 MB/s eta 0:00:00


In [4]:
!pip install torch torchvision pillow opencv-python

In [2]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

In [3]:
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

...

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
from google.colab import files
uploaded = files.upload()

Saving Screenshot 2025-08-01 152934.png to Screenshot 2025-08-01 152934.png


In [5]:
image_path = list(uploaded.keys())[0]

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image_path},
            {"type": "text", "text": "Describe this image in detail."}
        ],
    }
]

text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

image_inputs, video_inputs = process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
).to("cuda")

output_ids = model.generate(**inputs, max_new_tokens=150)

output = processor.batch_decode(
    output_ids[:, inputs.input_ids.shape[1]:],
    skip_special_tokens=True
)

print(output[0])

The image depicts a giant panda walking through a natural habitat. The panda is prominently featured in the center of the frame, occupying a significant portion of the image. It is walking on a grassy terrain with some patches of dry leaves and vegetation. The panda's fur is predominantly white with black patches on its face, ears, and paws. Its black ears are large and rounded, and its black eyes are prominent. The panda appears to be in a relaxed posture, with its front paws slightly raised, suggesting it might be drinking or sniffing the ground.

The background consists of a mix of grass, small plants, and some fallen leaves, indicating a natural, possibly forested area. The lighting suggests it is daytime, with sunlight filtering through


In [19]:
from google.colab import files
uploaded = files.upload()

Saving inputpic.jpg to inputpic.jpg


In [22]:
image_path = list(uploaded.keys())[0]

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image_path},
            {"type": "text", "text": "You are a vigilant observer tasked with documenting exactly what is happening in the current frame of video footage. Describe the scene as precisely and objectively as possible, focusing on visible actions, objects, people, positions, and interactions. Be concise and direct, but do not omit any details — even minor or seemingly irrelevant ones may be important later. Do not summarize across time or interpret intent. Just report what is visible in this single frame."}
        ],
    }
]

text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

image_inputs, video_inputs = process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
).to("cuda")

output_ids = model.generate(**inputs, max_new_tokens=150)

output = processor.batch_decode(
    output_ids[:, inputs.input_ids.shape[1]:],
    skip_special_tokens=True
)

print(output[0])

In the foreground, a man is pointing a gun directly at the camera. He is dressed in a black jacket and light-colored pants. Behind him, a group of uniformed soldiers is standing in formation, facing away from the camera. The soldiers are dressed in dark uniforms and appear to be in a disciplined formation. The background shows a paved road with a fence and some buildings, suggesting an urban setting. The overall atmosphere appears tense and serious.
